In [11]:
import chromadb
import ollama

In [12]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [13]:
# Equivalent to table creation in a relational database
collection = chroma_client.get_or_create_collection(name="company_policy")

In [14]:
documents = [
    "The standard health insurance plan covers dental up to $1500 annually after a $50 deductible.",
    "Employees are eligible for 20 days of paid time off (PTO) per calendar year, accrued monthly.",
    "Remote work requests must be submitted via the HR portal 14 days in advance for approval.",
    "The corporate expense policy allows up to $50 per day for meals during business travel."
]

# Generates a list: ['id_0', 'id_1', 'id_2', 'id_3']
doc_ids = [f"id_{i}" for i in range(len(documents))]

# ✅ THE CORRECT WAY: Pass the whole arrays at once. ChromaDB vectorizes them in parallel!
collection.add(
    documents=documents, 
    ids=doc_ids
)

In [15]:
# In a relational database, this would be the equivalent of inserting rows into a table. 
# Here, we are adding documents to our collection.
doc_ids = [f"id_{i}" for i in range(len(documents))]

In [16]:
for doc, id in zip(documents, doc_ids):
    collection.add(documents=doc, ids=id)

In [17]:
user_prompt = "What is the policy on working from home?"

In [6]:
search_results = collection.query(
    query_texts=[user_prompt],
    n_results=1
)

In [18]:
search_results 

{'ids': [['id_2']],
 'embeddings': None,
 'documents': [['Remote work requests must be submitted via the HR portal 14 days in advance for approval.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None]],
 'distances': [[1.4349734783172607]]}

In [7]:
retrieved_context = search_results["documents"][0][0]
print(f"🎯 Best Context Found in DB: \"{retrieved_context}\"")

🎯 Best Context Found in DB: "Remote work requests must be submitted via the HR portal 14 days in advance for approval."


In [8]:
system_prompt = "Answer the user's question based on the provided context."

In [9]:
output_response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Question: " + user_prompt + "\n\nContext:\n" + retrieved_context}
    ]
)

In [10]:
output_response.message.content

'Based on the provided context, it appears that there is a policy in place regarding remote work (working from home). However, the policy itself is not explicitly stated. The information provided does indicate that a request to work remotely must be submitted through an approved channel, specifically the HR portal, at least 14 days in advance for approval.\n\nThere is no direct answer to what the policy on working from home is, only that there is one and it requires approval via the HR portal with adequate notice.'